# Entrenamiento Transformer

En este notebook se entrena el transformer diseñado en transformer.py

In [1]:
from google.colab import drive
from google.colab import files
from tqdm.notebook import tqdm
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from torchvision.transforms import ToTensor
import sys
from sklearn.model_selection import train_test_split

drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es')


Mounted at /content/drive


In [2]:
from dataset import tiktoken
from dataset import Dataset
from dataset import Tokenizer
from transformer import Transformer

dataset_path = "/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es/dataset.json"
pesos_patht ="/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es/resultados/"
vocab_path = "/content/drive/MyDrive/TFG Matemáticas/TranslatorEn2Es/vocabulario.json"



Cargamos el dataset en un dataframe y dividimos 80% entrenamiento y 20% validacion

In [3]:
# Crear dataset y dataloader
df = pd.read_json(dataset_path)

train, val = train_test_split(
    df,
    test_size=0.8,
    random_state=42
)

print(len(train))
print(len(val))

200000
800000


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Thu Mar 12 16:52:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             54W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
tokenizer = Tokenizer(vocab_path)

train_loader  =  DataLoader(Dataset(train,tokenizer), batch_size=1250, shuffle=True)
val_loader = DataLoader(Dataset(val,tokenizer), batch_size=8, shuffle=False)
print(len(tokenizer))
transformer = Transformer(vocab_size=len(tokenizer),pad_id= tokenizer.pad_id()).to(device=device)


15000


ENTRENAMIENTO


In [7]:
import torch
from torch.nn import functional as F

@torch.no_grad()
def estimate_loss(eval_iters=5):
    out = {}
    transformer.eval() # Ponemos el modelo en modo evaluación (desactiva Dropout, etc.)

    loaders = {
        'train': train_loader,
        'val': val_loader
    }

    for split, loader in loaders.items():
        losses = torch.zeros(eval_iters)
        data_iter = iter(loader)

        for k in range(eval_iters):
            try:
                x_text, y_full_text = next(data_iter)
            except StopIteration:
                data_iter = iter(loader)
                x_text, y_full_text = next(data_iter)

            # 1. Tokenizamos igual que en el entrenamiento
            x_tokens = tokenizer.encode_batch(x_text).to(device)
            y_tokens = tokenizer.encode_batch(y_full_text).to(device)

            # 2. Desplazamiento (Teacher Forcing)
            y_input = y_tokens[:, :-1]
            y_target = y_tokens[:, 1:]

            # 3. Forward pass
            logits = transformer(x_tokens, y_input)

            # 4. Ajuste de dimensiones
            B, T, C = logits.shape
            logits = logits.reshape(B * T, C)
            y_target = y_target.reshape(B * T)

            # 5. Cálculo de la pérdida IGNORANDO el <PAD>
            loss = F.cross_entropy(logits, y_target, ignore_index=tokenizer.pad_id())
            losses[k] = loss.item()

        out[split] = losses.mean().item()

    transformer.train() # Volvemos a modo entrenamiento
    return out

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm

LEARNING_RATE = 1e-4
WARMUP_STEPS = 1000 # Durante los primeros 1000 batches, subiremos el LR de 0 a 1e-3
EVAL_ITERS = 1     # Épocas entre evaluaciones

model = transformer.to(device)
model.train()

# Definimos la pérdida ignorando el token <PAD>
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id())

# El optimizador
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
epochs = 10



for epoch in tqdm(range(epochs), desc="Training"):

    total_loss = 0
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{epochs}")


    if epoch % EVAL_ITERS == 0 or epoch == epochs - 1:

        losses = estimate_loss()
        torch.save(model.state_dict(), pesos_patht + f"{epoch}_val_{losses['val']:.4f}.pth")
        tqdm.write(f"Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

    for batch_idx, (x, y_full) in progress_bar:

        # --- FASE DE ENTRENAMIENTO ---
        x = tokenizer.encode_batch(x).to(device)
        y_full = tokenizer.encode_batch(y_full).to(device)

        # Desplazamos las secuencias
        y_input = y_full[:, :-1]
        y_target = y_full[:, 1:]

        # Forward pass
        logits = model(x, y_input)

        # Calcular Loss
        B, T, C = logits.shape
        logits = logits.reshape(B*T, C)
        y_target = y_target.reshape(B*T)
        loss = criterion(logits, y_target)

        total_loss += loss.item()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        # --- 2. GRADIENT CLIPPING ---
        # Corta los gradientes si son muy grandes para evitar que explote el modelo
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}'
        })

    # Al final de la época
    avg_loss = total_loss / len(train_loader)
    tqdm.write(f"Fin Epoch [{epoch+1}] | Loss Promedio: {avg_loss:.4f}\n")

Training:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1/10:   0%|          | 0/167 [00:00<?, ?it/s]

Train Loss: 3.9603 | Val Loss: 3.5403
Fin Epoch [1] | Loss Promedio: 3.8740



Epoch 2/10:   0%|          | 0/167 [00:00<?, ?it/s]

Train Loss: 3.6719 | Val Loss: 3.3171
Fin Epoch [2] | Loss Promedio: 3.6050



...

Train Loss: 2.9116 | Val Loss: 2.7385
Fin Epoch [6] | Loss Promedio: 2.9233



Epoch 7/10:   0%|          | 0/167 [00:00<?, ?it/s]

Train Loss: 2.8168 | Val Loss: 2.6574
Fin Epoch [7] | Loss Promedio: 2.7895



Epoch 8/10:   0%|          | 0/167 [00:00<?, ?it/s]

Train Loss: 2.5932 | Val Loss: 2.5789
